## Bibliotecas

In [115]:
import spacy
import pandas as pd
import re
import nltk
from pathlib import Path
from spacy.matcher import Matcher
from spacy.util import filter_spans
from nltk.corpus import stopwords
from pyvis.network import Network
import IPython



## 1. Dados

In [116]:
RAW = Path('../../data/raw')
INTERIM = Path('../../data/interim')
PROCESSED = Path('../../data/processed')

data = pd.read_csv(RAW / 'cases.csv')
metadata = pd.read_csv(RAW / 'metadata.csv')

In [117]:
# merge e seleção do text
full_data = pd.merge(data, metadata)
full_data = full_data[['case_text', 'gender', 'case_id', 'major_mesh_terms', 'mesh_terms']]

#mudar indice para selecionar outro caso
text = full_data['case_text'][0]

# modelo spaCy
nlp = spacy.load('en_core_web_sm')
doc = nlp(text)

nltk.download('stopwords', quiet=True)

True

In [118]:
# salva o dataset mesclado
full_data.to_csv(INTERIM / 'full_data_merged.csv', index=False)

## 2. Captura de Medidas (Regex)

In [137]:
#regex para extrair medidas e unidades
measurement_pattern = re.compile(r'(\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*[xX]\s*\d+(?:\.\d+)?)*|\d+(?:\.\d+)?(?:\s*[xX]\s*\d+(?:\.\d+)?)*)\s*(cm|mm|ng/ml|iu/ml|mg)')
measurements = []

#cria um dataframe com as medidas extraídas
for match in measurement_pattern.finditer(text):
    value, unit = match.group(1), match.group(2)
    measurements.append({
        'node_type': 'ExamResult',
        'label_original': match.group(0),
        'label_normalizado': f"{value} {unit}",
        'token_start': -1,
        'token_end': -1,
        'span_start': match.start(),
        'span_end': match.end(),
        'value': value,
        'unit': unit
    })

measurements_df = pd.DataFrame(measurements)

#salva as medidas como arquivos intermediários
measurements_df.to_csv(INTERIM / 'measurements_extracted.csv', index=False)

display(measurements_df)

,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,ExamResult,6cm,6 cm,-1,-1,337,340,6,cm
1,ExamResult,6cm,6 cm,-1,-1,516,519,6,cm
2,ExamResult,9cm,9 cm,-1,-1,522,525,9,cm
3,ExamResult,"12,476.5ng/ml","12,476.5 ng/ml",-1,-1,777,790,"12,476.5",ng/ml
4,ExamResult,6iu/ml,6 iu/ml,-1,-1,837,843,6,iu/ml
5,ExamResult,9.5cm,9.5 cm,-1,-1,2070,2075,9.5,cm
6,ExamResult,4.5cm,4.5 cm,-1,-1,2078,2083,4.5,cm
7,ExamResult,2.0cm,2.0 cm,-1,-1,2086,2091,2.0,cm


## 3. Extração de Entidades (scispaCy + Matcher) e Unificação de Nós

### Identificar paciente

In [120]:
extracted_entities = []

#lista de termos relacionados a pacientes para identifica-lo
patient_terms = ["woman", "man", "girl", "boy", "male", "female", "patient"]
patient_matcher = Matcher(nlp.vocab)
patient_matcher.add("PATIENT_DEMO", [[{"LOWER": {"IN": patient_terms}}]])
patient_matches = patient_matcher(doc)

#cria o nó do paciente com o label normalizado
patient_label = "patient"
if patient_matches:
    _, start, end = patient_matches[0]
    span = doc[start:end]
    patient_label = span.text.lower()
    extracted_entities.append({
        'node_type': 'Patient',
        'label_original': span.text,
        'label_normalizado': patient_label,
        'token_start': start, 'token_end': end,
        'span_start': span.start_char, 'span_end': span.end_char,
        'value': None, 'unit': None
    })
PATIENT_NODE_LABEL = patient_label

### Matcher para resgatar exames, procedimentos e conceitos clínicos

In [121]:
#lista de termos relacionados a doenças/sintomas
disease_keywords = [
    "pain", "nausea", "constipation", "malignancy", "neoplasm", "bleeding", 
    "cyst", "lesion", "abnormality", "gdc", "tumor", "mass", "carcinoma", 
    "adenocarcinoma", "melanoma", "sarcoma", "lymphoma", "leukemia", "metastasis", 
    "cancer", "nodule", "polyp", "ulcer", "inflammation", "infection", "abscess", 
    "sepsis", "fever", "chills", "fatigue", "weakness", "dizziness", "vertigo", 
    "syncope", "headache", "migraine", "dyspnea", "shortness of breath", "cough", 
    "wheezing", "apnea", "pneumonia", "asthma", "copd", "emphysema", "bronchitis", 
    "tuberculosis", "hypertension", "hypotension", "infarction", "ischemia", 
    "arrhythmia", "tachycardia", "bradycardia", "fibrillation", "heart failure", 
    "stroke", "aneurysm", "thrombosis", "embolism", "hemorrhage", "anemia", 
    "leukopenia", "thrombocytopenia", "coagulopathy", "diabetes", "hypothyroidism", 
    "hyperthyroidism", "obesity", "malnutrition", "anorexia", "diarrhea", 
    "vomiting", "dysphagia", "reflux", "gerd", "cirrhosis", "hepatitis", 
    "jaundice", "pancreatitis", "appendicitis", "cholecystitis", "gallstones", 
    "obstruction", "perforation", "rupture", "fistula", "stricture", "hernia", 
    "ileus", "arthritis", "osteoarthritis", "rheumatoid arthritis", "osteoporosis", 
    "fracture", "dislocation", "sprain", "strain", "myalgia", "arthralgia", 
    "neuropathy", "neuralgia", "seizure", "epilepsy", "dementia", "alzheimer", 
    "parkinson", "tremor", "paralysis", "paresis", "aphasia", "depression", 
    "anxiety", "schizophrenia", "bipolar disorder", "insomnia", "rash", 
    "erythema", "pruritus", "itching", "eczema", "psoriasis", "dermatitis", 
    "cellulitis", "edema", "swelling", "effusion", "ascites", "hematoma", 
    "bruise", "contusion", "laceration", "wound", "burn", "scar", "fibrosis", 
    "sclerosis", "atrophy", "hypertrophy", "hyperplasia", "dysplasia", 
    "syndrome", "disorder", "disease", "condition", "defect", "deformity", 
    "anomaly", "failure", "insufficiency", "dysfunction", "impairment", 
    "allergies", "anaphylaxis", "hypoxia", "hypoxemia", "acidosis", "alkalosis", 
    "hyperkalemia", "hypokalemia", "hyponatremia", "hypernatremia", "dehydration"
]


In [122]:
#lista de termos relacionados a exames médicos
exam_keywords = [
    "computed tomography", "ct", "ct scan", "cat scan", "magnetic resonance imaging", 
    "mri", "ultrasound", "sonogram", "echography", "x-ray", "radiograph", 
    "radiography", "pet scan", "positron emission tomography", "fluoroscopy", 
    "angiogram", "angiography", "mammogram", "mammography", "dexa scan", 
    "bone density scan", "electrocardiogram", "ecg", "ekg", "echocardiogram", 
    "echo", "electroencephalogram", "eeg", "electromyography", "emg", 
    "endoscopy", "colonoscopy", "sigmoidoscopy", "gastroscopy", 
    "oesophagogastroduodenoscopy", "bronchoscopy", "laryngoscopy", "cystoscopy", 
    "arthroscopy", "laparoscopy", "colposcopy", "hysteroscopy", "biopsy", 
    "fine needle aspiration", "fna", "lumbar puncture", "spinal tap", 
    "bone marrow aspiration", "bone marrow biopsy", "blood test", "complete blood count", 
    "cbc", "basic metabolic panel", "bmp", "comprehensive metabolic panel", "cmp", 
    "lipid panel", "liver function test", "lft", "thyroid function test", "tft", 
    "urinalysis", "urine culture", "blood culture", "stool test", "pcr", 
    "serology", "coagulation profile", "pt", "aptt", "inr", "d-dimer", 
    "troponin", "creatinine", "bun", "hba1c", "glucose test", "pap smear", 
    "amniocentesis", "spirometry", "pulmonary function test", "pft", "stress test", 
    "holter monitor", "tilt table test", "allergy test", "patch test", "skin prick test", 
    "resection", "pancreatectomy", "appendectomy", "cholecystectomy", "mastectomy", 
    "lumpectomy", "thyroidectomy", "prostatectomy", "hysterectomy", "oophorectomy", 
    "salpingectomy", "craniotomy", "lobectomy", "pneumonectomy", "gastrectomy", 
    "colectomy", "splenectomy", "nephrectomy", "cystectomy", "herniorrhaphy", 
    "arthroplasty", "osteotomy", "ligation", "bypass", "graft", "transplant", 
    "exploration", "examination", "physical examination", "palpation", "auscultation", 
    "percussion", "pathology", "cytology", "histology", "autopsy", "necropsy", 
    "smear", "swab", "culture", "karyotype", "genetic test", "sequencing", 
    "microarray", "flow cytometry", "chromatography", "spectrometry", "assay", 
    "titration", "dialysis", "hemodialysis", "plasmapheresis", "transfusion",
    "catheterization", "intubation", "ventilation", "defibrillation", "cardioversion",
    "pacemaker insertion", "stent placement", "angioplasty", "thrombectomy", 
    "embolectomy", "endarterectomy", "amputation", "debridement", "incision", 
    "drainage", "suturing", "cauterization", "laser therapy", "radiotherapy", 
    "chemotherapy", "immunotherapy", "physical therapy", "occupational therapy",
    "audiometry", "tympanometry", "fundoscopy", "tonometry", "visual acuity test",
    "neurological examination", "mental status examination", "glasgow coma scale",
    "apgar score", "treadmill test", "barium swallow", "barium enema", "dexa",
    "scintigraphy", "spect scan", "doppler", "elastography", "polysomnography",
    "sleep study", "sweat test", "mantoux test", "tuberculin skin test", "scan"
]

In [123]:
#definição de stopwords para filtrar palavras irrelevantes
stop_words_nltk = set(stopwords.words('english'))
custom_stops = {'day', 'history', 'month', 'year', 'time', 'presence', 'evidence', 'fig', 'figure'}
all_stopwords = stop_words_nltk.union(custom_stops)

concept_matcher = Matcher(nlp.vocab)
#padrão: adjetivo(0+) seguido de substantivo(1+)
concept_matcher.add("CLINICAL_CONCEPT", [
    [{"POS": "ADJ", "OP": "*"}, {"POS": "NOUN", "OP": "+"}],
    [{"POS": "PROPN", "OP": "+"}] # pega siglas 
])

#filtra sobreposições
spans = [doc[start:end] for _, start, end in concept_matcher(doc)]
filtered_spans = filter_spans(spans)

In [124]:
#extrai entidades e classifica com base em palavras-chave
for span in filtered_spans:

    # filtra tokens irrelevantes (stopwords, pontuação, dígitos)
    valid_tokens = [
        t for t in span
        if t.lemma_.lower() not in all_stopwords
        and not t.is_punct
        and not t.is_digit
    ]

    if not valid_tokens:
        continue

    clean_label = " ".join([t.lemma_.lower() for t in valid_tokens])

    if PATIENT_NODE_LABEL in clean_label:
        continue

    # Filtro de Negação
    negation_words = {"no", "not", "deny", "denies", "without", "negative", "absence"}
    start_token = span.start
    
    # Pega até 3 tokens antes da entidade para checar palavras de negação
    preceding_tokens = [t.lemma_.lower() for t in doc[max(0, start_token - 3):start_token]]
    
    is_negated = any(neg in preceding_tokens for neg in negation_words)
    if is_negated:
        continue 
        
    # Filtragem de termos genéricos
    # Ignora a palavra se o clean_label for exatamente ela sozinha
    generic_standalone_words = {"level", "cm x", "figs", "portion", "activity", "return", "follow", "cm", "mm", "ml", "mg", "ca", "cm cyst", "cyst cm", "diagnosis"}
    if clean_label in generic_standalone_words:
        continue

    # classificação baseada nas listas de palavras-chave (doenças e exames)
    node_class = 'MedicalConcept' # tipo genérico padrão
    if any(word in clean_label for word in disease_keywords):
        node_class = 'DISEASE'
    elif any(word in clean_label for word in exam_keywords):
        node_class = 'Procedure/Exam'

    extracted_entities.append({
        'node_type': node_class,
        'label_original': span.text,
        'label_normalizado': span.lemma_.lower(),
        'token_start': span.start, 'token_end': span.end,
        'span_start': span.start_char, 'span_end': span.end_char,
        'value': None, 'unit': None
    })

entities_df = pd.DataFrame(extracted_entities).drop_duplicates(subset=['label_normalizado']).reset_index(drop=True)

#salva as entidades médicas como arquivos intermediários
entities_df.to_csv(INTERIM / 'entities_extracted.csv', index=False)

display(entities_df.head(15))

,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,Patient,woman,woman,6,7,14,19,None,None
1,MedicalConcept,right flank,right flank,15,17,54,65,None,None
2,MedicalConcept,lower quadrant,low quadrant,18,20,70,84,None,None
3,DISEASE,abdominal pain,abdominal pain,20,22,85,99,None,None
4,DISEASE,nausea,nausea,24,25,116,122,None,None
5,DISEASE,constipation,constipation,26,27,127,139,None,None
6,MedicalConcept,family,family,32,33,159,165,None,None
7,MedicalConcept,medication history,medication history,34,36,170,188,None,None
8,Procedure/Exam,physical examination,physical examination,43,45,229,249,None,None
9,MedicalConcept,contrast,contrast,50,51,282,290,None,None


### Tabela de Nós Final

In [125]:
#concatena as entidades e medidas extraídas em um único dataframe de nós
final_nodes_df = pd.concat([entities_df, measurements_df], ignore_index=True)
final_nodes_df.insert(0, 'node_id', [f"N_{i+1:03d}" for i in range(len(final_nodes_df))])

#exporta os nós como arquivo final
final_nodes_df.to_csv(PROCESSED / 'nodes.csv', index=False)

print(f"TABELA DE NÓS")
display(final_nodes_df.head(15))

TABELA DE NÓS


,node_id,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,N_001,Patient,woman,woman,6,7,14,19,None,None
1,N_002,MedicalConcept,right flank,right flank,15,17,54,65,None,None
2,N_003,MedicalConcept,lower quadrant,low quadrant,18,20,70,84,None,None
3,N_004,DISEASE,abdominal pain,abdominal pain,20,22,85,99,None,None
4,N_005,DISEASE,nausea,nausea,24,25,116,122,None,None
5,N_006,DISEASE,constipation,constipation,26,27,127,139,None,None
6,N_007,MedicalConcept,family,family,32,33,159,165,None,None
7,N_008,MedicalConcept,medication history,medication history,34,36,170,188,None,None
8,N_009,Procedure/Exam,physical examination,physical examination,43,45,229,249,None,None
9,N_010,MedicalConcept,contrast,contrast,50,51,282,290,None,None


## 4. Geração do Grafo de Conhecimento (Tabela de Arestas)

In [126]:
#define relações entre verbos e tipos de arestas no grafo
verb_relations = {
    ("present", "have", "experience", "associate"): "HAS_SYMPTOM",
    ("undergo", "perform", "do", "plan", "convert"): "UNDERWENT_PROCEDURE",
    ("reveal", "demonstrate", "suggest", "show", "note", "observe"): "SUPPORTS",
    ("treat", "resect", "discharge"): "TREATED_BY"
}

### Arestas Sintáticas

In [127]:
edges = []
edge_id_counter = 1

#criação das arestas com base nos verbos e entidades extraídas
for sent in doc.sents:
    #pega apenas os nós que estão nesta frase
    nodes_in_sent = [row for _, row in entities_df.iterrows() if row['node_type'] != 'Patient' and sent.start <= row['token_start'] < sent.end]

    for token in sent:
        if token.pos_ == "VERB":
            verbo_lema = token.lemma_.lower()

            for verbs, rel in verb_relations.items():
                if verbo_lema in verbs:
                    if rel in ["HAS_SYMPTOM", "UNDERWENT_PROCEDURE", "TREATED_BY"]:
                        for node in nodes_in_sent:
                            # filtra um pouco
                            if rel == "HAS_SYMPTOM" and node['node_type'] not in ['DISEASE', 'MedicalConcept']: continue
                            if rel == "UNDERWENT_PROCEDURE" and node['node_type'] not in ['Procedure/Exam']: continue

                            edges.append({
                                'source_label': PATIENT_NODE_LABEL,
                                'target_label': node['label_normalizado'],
                                'relation': rel
                            })

                    elif rel == "SUPPORTS" and len(nodes_in_sent) >= 2:
                        edges.append({
                            'source_label': nodes_in_sent[0]['label_normalizado'],
                            'target_label': nodes_in_sent[1]['label_normalizado'],
                            'relation': rel
                        })
                    break

edges_df = pd.DataFrame(edges)
display(edges_df.head())

,source_label,target_label,relation
0,woman,right flank,HAS_SYMPTOM
1,woman,low quadrant,HAS_SYMPTOM
2,woman,abdominal pain,HAS_SYMPTOM
3,woman,nausea,HAS_SYMPTOM
4,woman,constipation,HAS_SYMPTOM


### Arestas de Valores por distância de caracteres

In [128]:
#criação das arestas entre entidades e medidas com base na proximidade no texto
for _, ent_row in entities_df.iterrows():
    if ent_row['node_type'] == 'Patient': continue
    for _, meas_row in measurements_df.iterrows():
        if abs(ent_row['span_start'] - meas_row['span_start']) < 30:
            edges.append({
                'source_label': ent_row['label_normalizado'],
                'target_label': meas_row['label_normalizado'],
                'relation': 'HAS_VALUE'
            })


## Grafo de Conhecimento final

In [129]:
final_graph_edges_df = pd.DataFrame(edges).drop_duplicates(subset=['source_label', 'target_label', 'relation']).reset_index(drop=True)

# Verifica se a coluna já existe antes de inserir para evitar o ValueError ao re-executar a célula
if 'edge_id' not in final_graph_edges_df.columns:
    final_graph_edges_df.insert(0, 'edge_id', [f"E_{i+1:03d}" for i in range(len(final_graph_edges_df))])

# exporta as arestas como arquivo final
final_graph_edges_df.to_csv(PROCESSED / 'edges.csv', index=False)

display(final_graph_edges_df.head(20))

,edge_id,source_label,target_label,relation
0,E_001,woman,right flank,HAS_SYMPTOM
1,E_002,woman,low quadrant,HAS_SYMPTOM
2,E_003,woman,abdominal pain,HAS_SYMPTOM
3,E_004,woman,nausea,HAS_SYMPTOM
4,E_005,woman,constipation,HAS_SYMPTOM
5,E_006,woman,computed tomography,UNDERWENT_PROCEDURE
6,E_007,contrast,computed tomography,SUPPORTS
7,E_008,woman,fna,UNDERWENT_PROCEDURE
8,E_009,woman,normal pancreatic echotexture,UNDERWENT_PROCEDURE
9,E_010,woman,internal septation,UNDERWENT_PROCEDURE


# Vizualização do Grafo (Com LLM)

In [130]:

# Inicializa a rede interativa configurada para o Colab
net = Network(notebook=False, directed=True, cdn_resources='remote', height="100vh", width="100%")

# Dicionário de cores para diferenciar visualmente o tipo de informação no grafo
color_map = {
    'Patient': '#79d2a6',        # Verde
    'DISEASE': '#ff4d4d',         # Vermelho
    'CHEMICAL': '#ffa64d',        # Laranja
    'Procedure/Exam': '#ffff66',  # Amarelo
    'MedicalConcept': '#66b3ff',  # Azul
    'ExamResult': '#d9d9d9'       # Cinza
}

# 1. Adicionando os Nós (Nodes)
for _, row in final_nodes_df.iterrows():
    node_id = row['label_normalizado']
    node_type = row['node_type']

    # Define a cor baseada no tipo, ou cinza claro por padrão
    node_color = color_map.get(node_type, '#e6e6e6')

    # Texto que aparece quando passa o mouse por cima
    hover_text = f"Tipo: {node_type}"
    if pd.notna(row.get('value')):
        hover_text += f" | Valor: {row['value']} {row['unit']}"

    net.add_node(node_id, label=node_id, title=hover_text, color=node_color)

# 2. Adicionando as Arestas (Edges/Relações)
# Criamos um conjunto de nós existentes para evitar erros caso a aresta aponte para um nó filtrado
existing_nodes = set(net.get_nodes())

for _, row in final_graph_edges_df.iterrows():
    source = row['source_label']
    target = row['target_label']
    relation = row['relation']

    if source in existing_nodes and target in existing_nodes:
        # Adiciona a seta com o rótulo da relação
        net.add_edge(source, target, title=relation, label=relation, color="#808080")

# 3. Física e Layout (Para o grafo não ficar todo embolado)
net.set_options("""
var options = {
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -50,
      "centralGravity": 0.01,
      "springLength": 100,
      "springConstant": 0.08
    },
    "minVelocity": 0.75,
    "solver": "forceAtlas2Based"
  },
  "edges": {
    "font": {
      "size": 10,
      "align": "middle"
    },
    "arrows": {
      "to": {"enabled": true, "scaleFactor": 0.5}
    }
  }
}
""")

# Gera o arquivo HTML e renderiza na célula do Colab
html_file = "../../assets/knowledge_graph.html"
net.show("../../assets/knowledge_graph.html", notebook=False)

../../assets/knowledge_graph.html


In [131]:
!ls ../..

README.md  assets  data  pipelines  requirements.txt  src


Couldn't find a suitable web browser!

Set the BROWSER environment variable to your desired browser.



Try running the update-desktop-database command. If you
don't have this command you should install the
desktop-file-utils package. This package is available from
http://freedesktop.org/wiki/Software/desktop-file-utils/
No applications found for mimetype: text/html
./bin/xdg-open: 882: x-www-browser: not found
/bin/xdg-open: 882: firefox: not found
/bin/xdg-open: 882: iceweasel: not found
/bin/xdg-open: 882: seamonkey: not found
/bin/xdg-open: 882: mozilla: not found
/bin/xdg-open: 882: epiphany: not found
/bin/xdg-open: 882: konqueror: not found
/bin/xdg-open: 882: chromium: not found
/bin/xdg-open: 882: chromium-browser: not found
/bin/xdg-open: 882: google-chrome: not found
/bin/xdg-open: 882: www-browser: not found
/bin/xdg-open: 882: links2: not found
/bin/xdg-open: 882: elinks: not found
/bin/xdg-open: 882: links: not found
/bin/xdg-open: 882: lynx: not found
/bin/xdg-open: 882: w3m: not found
xdg-open: no method available for opening '../../assets/knowledge_graph.html'
